# FlashGraph — Cloud GPU Build & Validation

This notebook:
1. Verifies GPU availability
2. Clones the FlashGraph repository
3. Builds the Python extension with CUDA support
4. Runs the full validation test suite
5. Benchmarks the fused kernel
6. Runs NVIDIA Nsight Compute profiling

**Requirements:** Google Colab with GPU runtime (T4, A100, or V100).
Go to Runtime → Change runtime type → GPU.

## Cell 1: Verify GPU

In [ ]:
!nvidia-smi
print()
!nvcc --version

## Cell 2: Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/Kevinbastin/flashgraph.git"
CLONE_DIR = "/content/flashgraph"

if os.path.exists(CLONE_DIR):
    print(f"Directory {CLONE_DIR} already exists, pulling latest...")
    !cd {CLONE_DIR} && git pull
else:
    !git clone {REPO_URL} {CLONE_DIR}

os.chdir(CLONE_DIR)
!ls -la

## Cell 3: Install Dependencies

In [ ]:
!pip install torch pytest --quiet

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:    {torch.version.cuda}")
    print(f"GPU device:      {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 4: Build FlashGraph Python Module

In [ ]:
!cd /content/flashgraph/python && pip install -e . 2>&1 | tail -20

# Verify import
import flashgraph
print("\n✓ flashgraph module imported successfully")
print(f"  Module location: {flashgraph.__file__}")

## Cell 5: Run Validation Tests

In [ ]:
!cd /content/flashgraph && python -m pytest tests/test_baseline.py -v --tb=short

## Cell 6: Quick Benchmark

Times the fused kernel using CUDA events for accurate GPU timing.

In [ ]:
import flashgraph
import torch

# Benchmark parameters
M, K, N = 128, 256, 512
WARMUP = 50
ITERATIONS = 1000

# Run benchmark
avg_ms = flashgraph.benchmark(M, K, N, WARMUP, ITERATIONS)

# Compute throughput
flops = 2 * M * K * N  # MatMul FLOPs (multiply-accumulate = 2 ops)
flops += M * K          # RMSNorm (approx)
flops += M * N * 10     # GELU (approx 10 ops per element)
tflops = flops / (avg_ms * 1e-3) / 1e12

print(f"=== FlashGraph Fused Kernel Benchmark ===")
print(f"  Dimensions:  M={M}, K={K}, N={N}")
print(f"  Precision:   FP16 compute, INT8 weights")
print(f"  Warmup:      {WARMUP} iterations")
print(f"  Measured:    {ITERATIONS} iterations")
print(f"  Avg time:    {avg_ms:.4f} ms")
print(f"  Throughput:  {tflops:.3f} TFLOPS")
print(f"  Latency:     {avg_ms * 1000:.1f} µs")

## Cell 7: NVIDIA Nsight Compute Profiling

### Roofline Analysis
Generates a roofline model to determine if the kernel is compute-bound or memory-bound.

In [ ]:
# Create a standalone profiling script
profiling_script = '''
import flashgraph
# Run benchmark with enough iterations for stable profiling
flashgraph.benchmark(M=128, K=256, N=512, warmup=10, iterations=20)
'''

with open('/content/flashgraph/profile_target.py', 'w') as f:
    f.write(profiling_script)

print("Profiling script written. Running ncu...")
print("(This may take 2-5 minutes)")

In [ ]:
# Roofline analysis
!ncu --set roofline \
     --kernel-name "fused_rmsnorm_matmul_gelu" \
     --launch-skip 5 --launch-count 3 \
     --target-processes all \
     -o /content/flashgraph_roofline \
     python /content/flashgraph/profile_target.py

In [ ]:
# Memory bandwidth and SM throughput analysis
!ncu --metrics \
     l1tex__t_bytes_pipe_lsu_mem_global_op_ld.sum.per_second,\
l1tex__t_bytes_pipe_lsu_mem_global_op_st.sum.per_second,\
sm__throughput.avg.pct_of_peak_sustained_elapsed,\
dram__throughput.avg.pct_of_peak_sustained_elapsed \
     --kernel-name "fused_rmsnorm_matmul_gelu" \
     --launch-skip 5 --launch-count 3 \
     python /content/flashgraph/profile_target.py

### Interpreting the Results

| Metric | Compute-Bound | Memory-Bound |
|--------|:---:|:---:|
| `sm__throughput` | > 60% | < 30% |
| `dram__throughput` | < 40% | > 60% |
| Roofline position | Above memory slope | On memory slope |

**If memory-bound:** Increase `TILE_K` in `fused_kernel.cu` to raise arithmetic intensity.

**If below both ceilings:** Occupancy issue — reduce register pressure with `--maxrregcount=64`.

**If compute-bound (target):** The kernel is fully utilizing the GPU's compute resources. ✓

In [ ]:
# View the generated report file
import os
report_file = "/content/flashgraph_roofline.ncu-rep"
if os.path.exists(report_file):
    size_mb = os.path.getsize(report_file) / 1e6
    print(f"✓ Roofline report generated: {report_file} ({size_mb:.1f} MB)")
    print("  Download this file and open in NVIDIA Nsight Compute GUI")
    print("  for the interactive roofline visualization.")
else:
    print("✗ Report file not found. Check ncu output above for errors.")